In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.patches as patches

import cv2

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2, ResNet50

import warnings
import os
from pathlib import Path
import time

warnings.filterwarnings('ignore')

print(f"GPU Kullanılabilir: {tf.config.list_physical_devices('GPU')}")
print(f"TensorFlow Sürümü: {tf.__version__}")


**VERİ SETİ YÜKLEME VE KEŞFETME**

In [ ]:
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

print(f"Eğitim veri seti şekli: {x_train.shape}")
print(f"Eğitim etiketleri şekli: {y_train.shape}")
print(f"Test veri seti şekli: {x_test.shape}")
print(f"Test etiketleri şekli: {y_test.shape}")

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

y_train = y_train.flatten()
y_test = y_test.flatten()

print(f"Sınıf Sayısı: {len(np.unique(y_train))}")
print(f"Sınıflar: {class_names}")


**VERİ SETİ GÖRSELLEŞTİRME**

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
axes = axes.ravel()

# 10 adet rastgele index seçilir
random_indices = np.random.choice(len(x_train), 10, replace=False)

for idx, ax in enumerate(axes):
    # Rastgele indexlerden görüntü alınır
    image = x_train[random_indices[idx]]
    label = y_train[random_indices[idx]]

    ax.imshow(image.astype('uint8'), cmap='gray')  # Fashion-MNIST tek kanallı olduğu için gri tonlamalı gösterilir

    ax.set_title(f"Sınıf: {class_names[label]}")
    ax.axis('off')

plt.tight_layout()
plt.show()

print("Örnek görüntüler yukarıda gösterilmektedir.")


**VERİ ÖN İŞLEME**

In [ ]:
# Görüntü piksel değerleri 0-255'ten 0-1 arasına ölçeklendirildi, bu şekilde daha hızlı ve stabil eğitilebilir
x_train_normalized = x_train.astype('float32') / 255.0
x_test_normalized = x_test.astype('float32') / 255.0

# Etiketler one-hot kodlamasına dönüştürülür
num_classes = len(class_names)
y_train_categorical = keras.utils.to_categorical(y_train, num_classes)
y_test_categorical = keras.utils.to_categorical(y_test, num_classes)

print(f"Normalize edilen eğitim verisi (min/max): {x_train_normalized.min():.2f} / {x_train_normalized.max():.2f}")
print(f"One Hot kodlanmış etiket örneği: {y_train_categorical[0]}")
print(f"One Hot kodlanmış etiket şekli: {y_train_categorical.shape}")


**VERİ AGUMENTASYON**

In [ ]:
# ImageDataGenerator, eğitim sırasında görüntüleri işler
# overfitting azalır, model daha iyi genelleme yapar
train_datagen = ImageDataGenerator(
    rotation_range=10,       # ±10 derece döndür
    width_shift_range=0.1,   # Genişlik için ±10% kaydır
    height_shift_range=0.1,  # Yükseklik için ±10% kaydır
    zoom_range=0.1,          # 0.9-1.1x arasında yakınlaş
    fill_mode='nearest'      # Boş pikselleri en yakın değerle doldur
)

# Test verisi için augmentasyon yapma (sadece normalize et)
test_datagen = ImageDataGenerator()

print("Veri Augmentasyon başarıyla tanımlandı.")


**CNN MODELİ OLUŞTURMA**

In [ ]:
# Sequential modeli oluştur (katmanlar sırayla eklenecek)
model_cnn = models.Sequential([
    # İlk Convolutional Blok
    # 32 filtre, 3x3 çekirdek, ReLU aktivasyon, input şekli 28x28x1 (Fashion-MNIST tek kanallı)
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),  # Boyutu 14x14 yapar
    layers.Dropout(0.25),

    # İkinci Convolutional Blok
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),  # Boyutu 7x7 yapar
    layers.Dropout(0.25),

    # Üçüncü Convolutional Blok
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2), padding='same'),  # Boyutu 4x4 yapar
    layers.Dropout(0.25),

    # Fully Connected Katmanlar
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),

    # 10 sınıf için output katmanı, softmax aktivasyon
    layers.Dense(num_classes, activation='softmax')
])

# Model mimarisini göster
model_cnn.summary()


**MODEL DERLEME VE EĞİTİMİ**

In [ ]:
# Modeli açıkça derlemeden önce yapılandırılır
# input_shape, batch boyutunu (None) içermelidir
model_cnn.build(input_shape=(None, 28, 28, 1))

model_cnn.compile(optimizer='adam',
                   loss='categorical_crossentropy',
                   metrics=['accuracy'])
print("Model derlendi.")


In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,  # 5 epoch
    restore_best_weights=True  # En iyi olanları geri yükle
)

# ReduceLROnPlateau, validation_loss iyileşmezse öğrenme oranını azaltır
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,  # Öğrenme oranını 0.5 ile çarp
    patience=3,  # 3 epoch
    min_lr=1e-7  # Minimum öğrenme oranı
)

# Modeli eğit
print("Model eğitimi başlıyor...")
start_time = time.time()

# Conv2D 4 boyutlu girdi beklediği için kanal ekseni ekleniyor: (N, 28, 28) -> (N, 28, 28, 1)
x_train_processed = np.expand_dims(x_train_normalized, axis=-1)

history = model_cnn.fit(
    x_train_processed, y_train_categorical,
    batch_size=128,       # Her adımda 128 örnek
    epochs=30,            # Maksimum 30 epoch
    validation_split=0.2, # %20'si validation için ayrılır
    callbacks=[early_stopping, reduce_lr],
    verbose=1             # Her epoch'u yazdır
)

training_time = time.time() - start_time
print(f"\nEğitim süresi: {training_time:.2f} saniye")


**EĞİTİM SONUÇLARINI GÖRSELLEŞTİRME**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Accuracy grafik
axes[0].plot(history.history['accuracy'], label='Training Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss grafik
axes[1].plot(history.history['loss'], label='Training Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Model Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

final_accuracy = history.history['accuracy'][-1]
final_val_accuracy = history.history['val_accuracy'][-1]
print(f"\nFinal Training Accuracy: {final_accuracy:.4f}")
print(f"Final Validation Accuracy: {final_val_accuracy:.4f}")


**TEST SETİ İLE DEĞERLENDİRME**

In [ ]:
# Test seti üzerinde modeli değerlendir
# Eğitimle aynı şekilde kanal ekseni eklenir: (N, 28, 28) -> (N, 28, 28, 1)
x_test_processed = np.expand_dims(x_test_normalized, axis=-1)

test_loss, test_accuracy = model_cnn.evaluate(x_test_processed, y_test_categorical, verbose=0)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Set Başarısı: {test_accuracy*100:.2f}%")


**MODEL TAHMİNLERİ VE SONUÇLAR**

In [ ]:
# Test seti üzerinde tahmin yap
y_pred_probs = model_cnn.predict(x_test_processed, verbose=0)

# Maksimum probability'ye sahip sınıfı seç
y_pred = np.argmax(y_pred_probs, axis=1)

# 20 rastgele test örneği seç ve sonuçları göster
fig, axes = plt.subplots(4, 5, figsize=(15, 12))
axes = axes.ravel()

random_indices = np.random.choice(len(x_test), 20, replace=False)

for idx, ax in enumerate(axes):
    image = x_test[random_indices[idx]]
    true_label = y_test[random_indices[idx]]
    pred_label = y_pred[random_indices[idx]]
    confidence = y_pred_probs[random_indices[idx]][pred_label]

    ax.imshow(image.astype('uint8'), cmap='gray')

    color = 'green' if true_label == pred_label else 'red'
    title = f"Tahmin: {class_names[pred_label]}\n"
    title += f"Gerçek: {class_names[true_label]}\n"
    title += f"Güven: {confidence:.2f}"

    ax.set_title(title, color=color, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()


**CONFUSSION MATRIX VE SINIFLANDIRMA RAPORU**

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
plt.imshow(cm, cmap='Blues', interpolation='nearest')
plt.title('Confusion Matrix')
plt.colorbar()

tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=45, ha='right')
plt.yticks(tick_marks, class_names)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center',
                  color='white' if cm[i, j] > cm.max() / 2 else 'black')

plt.ylabel('Gerçek Sınıf')
plt.xlabel('Tahmin Sınıfı')
plt.tight_layout()
plt.show()


In [ ]:
# Sınıflandırma Raporu: Precision, Recall, F1-Score metrikleri
report = classification_report(y_test, y_pred, target_names=class_names)
print(report)


**TRANSFER LEARNING - MobileNetV2 İLE CNN **

In [ ]:
# ImageNet verisi ile önceden eğitilmiş MobileNetV2 yükle
# include_top=False: Classification katmanını hariç tut, sadece feature extraction
base_model = MobileNetV2(
    input_shape=(32, 32, 3),
    include_top=False,
    weights='imagenet'
)

# Base model'in katmanlarını freeze et (eğitim sırasında ağırlıkları değiştirme)
base_model.trainable = False

# Transfer Learning modeli oluştur
model_transfer = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

model_transfer.summary()


In [ ]:
# Transfer Learning modeli derle
model_transfer.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Transfer Learning modeli derlendi.")


In [ ]:
# MobileNetV2, 32x32x3 boyutunda girdi beklediği için Fashion-MNIST görüntüleri
# önce 32x32'ye yeniden boyutlandırılır, sonra tek kanaldan 3 kanala (RGB) çevrilir
x_train_for_transfer = tf.image.resize(x_train_processed, [32, 32])
x_train_for_transfer = tf.image.grayscale_to_rgb(x_train_for_transfer).numpy()

x_test_for_transfer = tf.image.resize(x_test_processed, [32, 32])
x_test_for_transfer = tf.image.grayscale_to_rgb(x_test_for_transfer).numpy()

print("Transfer Learning modeli eğitiliyor...")
start_time = time.time()

history_transfer = model_transfer.fit(
    x_train_for_transfer, y_train_categorical,
    batch_size=128,
    epochs=15,  # Daha az epoch yeterli
    validation_split=0.2,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

training_time_transfer = time.time() - start_time
print(f"\nTransfer Learning eğitim süresi: {training_time_transfer:.2f} saniye")


In [ ]:
# Transfer Learning modeli test et
test_loss_transfer, test_accuracy_transfer = model_transfer.evaluate(x_test_for_transfer, y_test_categorical, verbose=0)

print(f"Transfer Learning Test Accuracy: {test_accuracy_transfer:.4f}")
print(f"Transfer Learning Test Set Başarısı: {test_accuracy_transfer*100:.2f}%")

# Modelleri karşılaştır
print(f"\n--- Model Karşılaştırması ---")
print(f"CNN Model Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Transfer Learning Test Accuracy: {test_accuracy_transfer:.4f} ({test_accuracy_transfer*100:.2f}%)")
print(f"Transfer Learning Avantajı: {(test_accuracy_transfer - test_accuracy)*100:.2f}%")
